In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio

if 'vscode' in pio.renderers:
    pio.renderers.default = 'vscode'
else:
    pio.renderers.default = 'notebook'

cwd = Path.cwd().resolve()
repo_candidates = [cwd, *cwd.parents]
repo_root = next((p for p in repo_candidates if (p / 'src').exists() and (p / 'data').exists()), cwd)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

clean_path = repo_root / 'data' / 'processed' / 'ibrd_clean.csv'
if not clean_path.exists():
    from src.data_pipeline.silver_layer import clean_ibrd_data
    clean_ibrd_data(
        raw_path=repo_root / 'data' / 'raw' / 'ibrd_synthetic.csv',
        silver_path=clean_path,
    )

df = pd.read_csv(clean_path)

output_dir = repo_root / 'notebooks' / 'exports'
output_dir.mkdir(parents=True, exist_ok=True)


def show_and_save(fig, filename):
    fig.show(renderer='vscode')
    try:
        export_path = output_dir / f'{filename}.png'
        fig.write_image(export_path, width=1400, height=900, scale=2)
        print(f'Saved chart: {export_path}')
    except Exception as exc:
        print(f'PNG export skipped: {exc}')


def calculate_risk_score(row):
    score = 0

    original_amount = row['Original Principal Amount (US$)']
    due_amount = row['Due to IBRD (US$)']

    if pd.notna(original_amount) and original_amount != 0:
        if due_amount / original_amount > 0.5:
            score += 2

    if row['repayment_ratio'] < 0.2 and row['loan_age_years'] > 10:
        score += 2
    if pd.notna(row['is_cancelled']) and row['is_cancelled']:
        score += 1
    if row['Due to IBRD (US$)'] > 100_000_000:
        score += 1
    return score


df['risk_score'] = df.apply(calculate_risk_score, axis=1)

risk_counts = df['risk_score'].value_counts().sort_index()
fig = px.bar(x=risk_counts.index, y=risk_counts.values, title='Loan Risk Score Distribution', labels={'x': 'Risk Score', 'y': 'Number of Loans'})
show_and_save(fig, 'loan_risk_score_distribution')

high_risk = df[df['risk_score'] >= 3]
fig = px.bar(high_risk.groupby('Region').size().reset_index(name='count'), x='Region', y='count', title='High-Risk Loans by Region')
show_and_save(fig, 'high_risk_loans_by_region')

Saved chart: /home/rigii/ATA/notebooks/exports/loan_risk_score_distribution.png


Saved chart: /home/rigii/ATA/notebooks/exports/high_risk_loans_by_region.png


In [8]:
# AWR-style portfolio risk overview
risk_by_region = (
    df.groupby('Region', as_index=False)
    .agg(
        avg_risk=('risk_score', 'mean'),
        total_outstanding=('Due to IBRD (US$)', 'sum'),
        loans=('Loan Number', 'count'),
    )
    .sort_values('avg_risk', ascending=False)
)

fig = px.bar(
    risk_by_region,
    x='Region',
    y='avg_risk',
    color='total_outstanding',
    title='AWR: Average Portfolio Risk by Region',
    labels={'avg_risk': 'Average Risk Score', 'total_outstanding': 'Outstanding (US$)'},
)
show_and_save(fig, 'awr_average_risk_by_region')

age_bins = pd.cut(
    df['loan_age_years'],
    bins=[0, 5, 10, 20, 30, 50, 100],
    labels=['0-5', '5-10', '10-20', '20-30', '30-50', '50+'],
)
age_risk = (
    df.assign(age_bucket=age_bins)
    .groupby('age_bucket', as_index=False)['risk_score']
    .mean()
    .sort_values('age_bucket')
)

fig = px.line(
    age_risk,
    x='age_bucket',
    y='risk_score',
    markers=True,
    title='AWR: Risk Score by Loan Age',
    labels={'age_bucket': 'Loan Age (Years)', 'risk_score': 'Average Risk Score'},
)
show_and_save(fig, 'awr_risk_by_loan_age')

risk_summary = {
    'mean_risk_score': round(df['risk_score'].mean(), 2),
    'median_risk_score': round(df['risk_score'].median(), 2),
    'high_risk_loans': int((df['risk_score'] >= 3).sum()),
    'share_high_risk': round((df['risk_score'] >= 3).mean() * 100, 2),
}
risk_summary


Saved chart: /home/rigii/ATA/notebooks/exports/awr_average_risk_by_region.png


Saved chart: /home/rigii/ATA/notebooks/exports/awr_risk_by_loan_age.png


{'mean_risk_score': np.float64(1.8),
 'median_risk_score': np.float64(2.0),
 'high_risk_loans': 455,
 'share_high_risk': np.float64(37.92)}

In [7]:
# Quick interpretation
print('AWR risk summary:')
print(risk_summary)
print('Top regions by average risk:')
print(risk_by_region.head().to_string(index=False))
print('Age buckets with elevated risk:')
print(age_risk.sort_values('risk_score', ascending=False).head().to_string(index=False))


AWR risk summary:
{'mean_risk_score': np.float64(1.8), 'median_risk_score': np.float64(2.0), 'high_risk_loans': 455, 'share_high_risk': np.float64(37.92)}
Top regions by average risk:
               Region  avg_risk  total_outstanding  loans
            East Asia  1.979839       3.303893e+10    248
               Africa  1.846774       3.210225e+10    248
        Latin America  1.791822       3.249264e+10    269
Europe & Central Asia  1.750000       2.696456e+10    212
           South Asia  1.605381       2.767130e+10    223
Age buckets with elevated risk:
age_bucket  risk_score
     10-20    1.955850
     20-30    1.923304
       0-5    1.545455
      5-10    1.511811
